# QLoRA 8 GB feasibility test: Llama 3.2 3B on a simulated 8 GB card

**Question this answers:** can our QLoRA training fit on an 8 GB GPU (the RTX 4060), using **our own training rows**
(`poc_thesis/training/train.jsonl`)?

**How:** Colab's T4 has about 15 GB. Each test run caps PyTorch at **7.0 GB** (`torch.cuda.set_per_process_memory_fraction`),
leaving about 1 GB of an 8 GB card for the driver and the display. If a run finishes under the cap, that setting fits.
Memory use depends on the software settings, not the GPU's speed, so this is a fair stand-in. The T4 is slightly *stricter*:
it cannot use FlashAttention, and it uses fp16, which takes the same memory as the RTX 4060's bf16.

**Steps**
1. `Runtime → Change runtime type → T4 GPU`, then run the cells in order.
2. Hugging Face: accept the Llama 3.2 license once at https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct,
   and add your token as a Colab secret named `HF_TOKEN` (key icon on the left). The token is never printed.
3. Upload `poc_thesis/training/train.jsonl` when asked.
4. Read the results table (step 7), then optionally run the learning check (step 8).

**What each run does:** loads Llama 3.2 3B Instruct in 4-bit NF4 (double quantization), adds LoRA to all 7 linear layers,
turns on gradient checkpointing, and trains a few steps with batch size 1, gradient accumulation, and 8-bit AdamW.
The loss covers only the answer, not the passages. The input is the **longest** real row (worst case), or that row
lengthened with **real passages** from the other rows to stand in for 5 passages.

Not a quality result, and nothing here goes in the manuscript as a finding. It's a development check (the "smoke test").

## 1. Check the GPU

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

## 2. Install the training libraries (about 1–2 minutes)

In [ ]:
!pip -q install -U transformers peft bitsandbytes accelerate liger-kernel
import importlib.metadata as md
for p in ["torch", "transformers", "peft", "bitsandbytes", "accelerate", "liger-kernel"]:
    try:
        print(f"{p:14} {md.version(p)}")
    except md.PackageNotFoundError:
        print(f"{p:14} not installed")

## 3. Hugging Face access and model download
If the download says the model is gated, accept the license at the link in the first cell with the same account as your token.

In [ ]:
import os
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"   # the study's model (gated: accept the license first)

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("Using the HF_TOKEN Colab secret.")
except Exception:
    from huggingface_hub import login
    login()   # paste the token when asked; it is not printed

from huggingface_hub import snapshot_download
path = snapshot_download(MODEL_ID, allow_patterns=["*.json", "*.safetensors", "tokenizer*"])
print("Model files ready:", path)

## 4. Upload our training rows (`poc_thesis/training/train.jsonl`)

In [ ]:
import json, os
from google.colab import files

if not os.path.exists("train.jsonl"):
    print("Choose poc_thesis/training/train.jsonl")
    uploaded = files.upload()
    name = next(iter(uploaded))
    if name != "train.jsonl":
        os.replace(name, "train.jsonl")

rows = [json.loads(l) for l in open("train.jsonl", encoding="utf-8")]
assert rows and all({"id", "type", "prompt", "completion"} <= set(r) for r in rows), "not our train.jsonl format"

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)
lengths = {}
for r in rows:
    text = tok.apply_chat_template(r["prompt"] + r["completion"], tokenize=False)
    lengths[r["id"]] = len(tok(text, add_special_tokens=False)["input_ids"])
    print(f'{r["id"]:9} {r["type"]:30} {lengths[r["id"]]:5d} tokens')
print(f"\n{len(rows)} rows · longest {max(lengths.values())} tokens · mean {sum(lengths.values()) / len(lengths):.0f} tokens")

## 5. Test settings
The defaults test 3 lengths × 2 LoRA ranks × 2 loss modes = **12 runs, about 25–40 minutes**. Remove entries to go faster.
- `standard`: the plain Hugging Face QLoRA recipe.
- `liger`: the same, plus Liger's fused cross-entropy. This is the same loss computed in pieces, so the full
  128k-vocabulary output table is never held in memory at once.

In [ ]:
CAP_GB = 7.0            # the simulated 8 GB card: 7 GB for PyTorch + ~1 GB for driver and display
LENGTHS = {             # label -> target tokens (None = the longest real row as-is)
    "3 passages (real)": None,
    "5 passages (stand-in)": 2600,
    "stress test": 3100,
}
RANKS = [16, 64]
MODES = ["standard", "liger"]
STEPS, ACCUM = 6, 2     # 6 forward/backward passes = 3 optimizer steps; the peak is reached within these

## 6. The run script (one run per process, so an out-of-memory crash can't affect the next run)

In [ ]:
%%writefile feas_run.py
"""One feasibility run: QLoRA training steps on one worst-case example, under a memory cap.
Called by the notebook as: python feas_run.py '<json config>'. Prints one line: RESULT {...}"""
import json, os, re, sys, time

cfg = json.loads(sys.argv[1])
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch

# ---- the simulated 8 GB card: cap what PyTorch may use on this GPU
total = torch.cuda.get_device_properties(0).total_memory
torch.cuda.set_per_process_memory_fraction(min(1.0, cfg["cap_gb"] * 1024**3 / total), 0)

result = {k: cfg[k] for k in ("length", "rank", "mode")}
result.update(status="not run", tokens=None, peak_alloc_gb=None, peak_reserved_gb=None, tok_per_s=None)


def done(**kw):
    result.update(kw)
    print("RESULT " + json.dumps(result), flush=True)
    sys.exit(0)


def gb(x):
    return round(x / 1024**3, 2)


if cfg["mode"] == "liger":                       # fused linear cross-entropy: never builds the full 128k-vocab logits
    try:
        from liger_kernel.transformers import apply_liger_kernel_to_llama
        apply_liger_kernel_to_llama(rope=True, rms_norm=True, swiglu=True,
                                    cross_entropy=False, fused_linear_cross_entropy=True)
    except Exception as e:
        done(status=f"liger unavailable: {type(e).__name__}: {e}"[:200])

import bitsandbytes as bnb
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

cdtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16   # T4: fp16; RTX 4060: bf16

# ---- data: the longest real row, optionally lengthened with real passages from the other rows
rows = [json.loads(l) for l in open(cfg["data"], encoding="utf-8")]
tok = AutoTokenizer.from_pretrained(cfg["model_id"])


def passages_of(row):
    body = row["prompt"][1]["content"].split("Passages:\n\n", 1)[1].split("\n\nQuestion: ", 1)[0]
    return re.split(r"\n\n(?=\[Passage \d+\] )", body)


def encode(prompt, completion):
    p_text = tok.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
    f_text = tok.apply_chat_template(prompt + completion, tokenize=False)
    p_ids = tok(p_text, add_special_tokens=False)["input_ids"]
    f_ids = tok(f_text, add_special_tokens=False)["input_ids"]
    n = len(p_ids)                                                    # prompt tokens get no loss
    return f_ids, [-100] * n + f_ids[n:]


enc = [encode(r["prompt"], r["completion"]) for r in rows]
base_i = max(range(len(rows)), key=lambda i: len(enc[i][0]))
ids, labels = enc[base_i]
if cfg["target_tokens"]:
    row = rows[base_i]
    donors = [p for j, r in enumerate(rows) if j != base_i for p in passages_of(r)]
    user = row["prompt"][1]["content"]
    head, question = user.rsplit("\n\nQuestion: ", 1)
    n_given = len(passages_of(row))
    k = 0
    while len(ids) < cfg["target_tokens"] and k < 4 * len(donors):
        text = re.sub(r"^\[Passage \d+\]", f"[Passage {n_given + k + 1}]", donors[k % len(donors)])
        head = head + "\n\n" + text
        k += 1
        prompt = [row["prompt"][0], {"role": "user", "content": head + "\n\nQuestion: " + question}]
        ids, labels = encode(prompt, row["completion"])
result.update(tokens=len(ids), base_row=rows[base_i]["id"])

# ---- model: the standard QLoRA recipe
try:
    bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=cdtype)
    kw = dict(quantization_config=bnb_cfg, device_map={"": 0})
    try:
        model = AutoModelForCausalLM.from_pretrained(cfg["model_id"], dtype=cdtype, **kw)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(cfg["model_id"], torch_dtype=cdtype, **kw)
    model.config.use_cache = False
    try:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                                gradient_checkpointing_kwargs={"use_reentrant": False})
    except TypeError:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model = get_peft_model(model, LoraConfig(
        r=cfg["rank"], lora_alpha=2 * cfg["rank"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    result.update(trainable_params=trainable, weights_loaded_gb=gb(torch.cuda.memory_allocated()))

    opt = bnb.optim.AdamW8bit([p for p in model.parameters() if p.requires_grad], lr=2e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=(cdtype == torch.float16))
    x = torch.tensor([ids], device="cuda")
    y = torch.tensor([labels], device="cuda")
    model.train()
    torch.cuda.reset_peak_memory_stats()
    times, losses = [], []
    for step in range(cfg["steps"]):
        t0 = time.time()
        with torch.autocast("cuda", dtype=cdtype):
            loss = model(input_ids=x, labels=y).loss
        scaler.scale(loss / cfg["accum"]).backward()
        if (step + 1) % cfg["accum"] == 0:
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
        torch.cuda.synchronize()
        times.append(time.time() - t0)
        losses.append(round(float(loss), 4))
except torch.cuda.OutOfMemoryError:
    done(status="OOM", peak_alloc_gb=gb(torch.cuda.max_memory_allocated()),
         peak_reserved_gb=gb(torch.cuda.max_memory_reserved()))
except Exception as e:
    done(status=f"error: {type(e).__name__}: {e}"[:300])

steady = sorted(times[1:])[len(times[1:]) // 2] if len(times) > 1 else times[0]
done(status="ok", peak_alloc_gb=gb(torch.cuda.max_memory_allocated()),
     peak_reserved_gb=gb(torch.cuda.max_memory_reserved()),
     sec_per_step=round(steady, 2), tok_per_s=round(len(ids) / steady), losses=losses,
     compute_dtype=str(cdtype).replace("torch.", ""))

## 7. Run the memory test and read the results

In [ ]:
import itertools, json, os, subprocess, sys, time
import pandas as pd

results = []
for (label, target), rank, mode in itertools.product(LENGTHS.items(), RANKS, MODES):
    cfg = dict(length=label, target_tokens=target, rank=rank, mode=mode, cap_gb=CAP_GB,
               steps=STEPS, accum=ACCUM, model_id=MODEL_ID, data="train.jsonl")
    t0 = time.time()
    p = subprocess.run([sys.executable, "feas_run.py", json.dumps(cfg)], capture_output=True, text=True,
                       env={**os.environ, "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})
    line = next((l for l in p.stdout.splitlines() if l.startswith("RESULT ")), None)
    res = json.loads(line[7:]) if line else dict(length=label, rank=rank, mode=mode,
                                                 status="crashed: " + p.stderr.strip()[-300:])
    res["wall_s"] = round(time.time() - t0)
    results.append(res)
    print(f'{label:22} r={rank:<3} {mode:9} -> {res["status"]:10} tokens={res.get("tokens")} '
          f'peak={res.get("peak_reserved_gb")} GB  ({res["wall_s"]} s)')

df = pd.DataFrame(results)
for c in ("peak_reserved_gb", "tok_per_s", "tokens"):
    df[c] = pd.to_numeric(df.get(c), errors="coerce")
df["fits_8gb"] = (df["status"] == "ok") & (df["peak_reserved_gb"] <= CAP_GB)
df["headroom_gb"] = (CAP_GB - df["peak_reserved_gb"]).round(2)
cols = ["length", "tokens", "rank", "mode", "status", "peak_reserved_gb", "headroom_gb", "tok_per_s", "fits_8gb"]
df.to_csv("feasibility_results.csv", index=False)
df[[c for c in cols if c in df]]

In [ ]:
# Plain-language verdict + a rough training-time estimate from the measured speed
ok = df[df["fits_8gb"]]
print(f"Cap used: {CAP_GB} GB for PyTorch (simulating an 8 GB card with ~1 GB left for driver and display).\n")
for label in LENGTHS:
    sub = df[df["length"] == label]
    good = sub[sub["fits_8gb"]]
    if len(good) == len(sub):
        print(f"✅ {label}: every setting fits. Worst peak {sub['peak_reserved_gb'].max()} GB.")
    elif len(good):
        best = good.sort_values("peak_reserved_gb").iloc[0]
        print(f"⚠️ {label}: fits only with some settings; the lowest peak is rank {best['rank']} + {best['mode']} "
              f"at {best['peak_reserved_gb']} GB.")
    else:
        print(f"❌ {label}: no tested setting fits under {CAP_GB} GB.")

std = ok[(ok["length"] == "3 passages (real)") & (ok["mode"] == "standard")]
if len(std):
    tps = std["tok_per_s"].min()
    mean_tokens = sum(lengths.values()) / len(lengths)
    minutes = 300 * mean_tokens / tps / 60
    print(f"\nSpeed on this T4: about {tps} tokens/s. For 300 examples of ~{mean_tokens:.0f} tokens, one epoch ≈ "
          f"{minutes:.0f} min here. The RTX 4060 will differ, so treat this as a rough guide only.")
print("\nSaved: feasibility_results.csv (download it from the Files panel on the left).")

## 8. Optional: learning check (about 5–10 minutes)
This trains a tiny adapter on the rows in `train.jsonl` under the same 7 GB cap, then compares answers **before** (unadapted)
and **after** (QLoRA-adapted) on the same prompts. With only a handful of rows the adapter memorises them, so this proves
only that the pipeline works end to end: the loss falls, the answer format appears, and refusals come out as the decline
sentence. It is **not** a measure of quality.

In [ ]:
%%writefile learn_check.py
"""Learning check: train a tiny adapter on the sample rows under the same cap, then generate.
With 5 rows the adapter memorises them. This only proves the pipeline works end to end
(loss falls, outputs take the trained format). It is NOT a quality result."""
import json, os, sys, time

cfg = json.loads(sys.argv[1])
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch

total = torch.cuda.get_device_properties(0).total_memory
torch.cuda.set_per_process_memory_fraction(min(1.0, cfg["cap_gb"] * 1024**3 / total), 0)

import bitsandbytes as bnb
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

DECLINE = "The provided manual passages do not answer this question."
cdtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
rows = [json.loads(l) for l in open(cfg["data"], encoding="utf-8")]
tok = AutoTokenizer.from_pretrained(cfg["model_id"])


def encode(prompt, completion):
    p_ids = tok(tok.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True),
                add_special_tokens=False)["input_ids"]
    f_ids = tok(tok.apply_chat_template(prompt + completion, tokenize=False),
                add_special_tokens=False)["input_ids"]
    return f_ids, [-100] * len(p_ids) + f_ids[len(p_ids):]


bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=cdtype)
kw = dict(quantization_config=bnb_cfg, device_map={"": 0})
try:
    model = AutoModelForCausalLM.from_pretrained(cfg["model_id"], dtype=cdtype, **kw)
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(cfg["model_id"], torch_dtype=cdtype, **kw)
tok.pad_token = tok.pad_token or tok.eos_token


def generate(m, prompt):
    text = tok.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
    x = tok(text, add_special_tokens=False, return_tensors="pt").to("cuda")
    with torch.no_grad(), torch.autocast("cuda", dtype=cdtype):
        out = m.generate(**x, max_new_tokens=cfg["max_new_tokens"], do_sample=False,
                         pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def verdict(row, text):
    target = row["completion"][0]["content"]
    if target == DECLINE:
        return "declined correctly" if text.startswith(DECLINE) else "should have declined"
    if text.startswith(DECLINE):
        return "declined, but should answer"
    return "answered with source lines" if "Source: " in text else "answered, no source lines"


print("BEFORE training (unadapted, same prompts):")
model.config.use_cache = True
before = {r["id"]: generate(model, r["prompt"]) for r in rows}
for r in rows:
    print(f'  {r["id"]:9} {r["type"]:30} -> {verdict(r, before[r["id"]])}')

model.config.use_cache = False
try:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                            gradient_checkpointing_kwargs={"use_reentrant": False})
except TypeError:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=cfg["rank"], lora_alpha=2 * cfg["rank"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
opt = bnb.optim.AdamW8bit([p for p in model.parameters() if p.requires_grad], lr=cfg["lr"])
scaler = torch.amp.GradScaler("cuda", enabled=(cdtype == torch.float16))
data = [encode(r["prompt"], r["completion"]) for r in rows]

model.train()
torch.cuda.reset_peak_memory_stats()
t0, losses = time.time(), []
for epoch in range(cfg["epochs"]):
    ep = []
    for ids, labels in data:
        with torch.autocast("cuda", dtype=cdtype):
            loss = model(input_ids=torch.tensor([ids], device="cuda"),
                         labels=torch.tensor([labels], device="cuda")).loss
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        ep.append(float(loss))
    losses.append(sum(ep) / len(ep))
    print(f"  epoch {epoch + 1:2d}  mean loss {losses[-1]:.3f}")
print(f"Training: {time.time() - t0:.0f} s, peak reserved {torch.cuda.max_memory_reserved() / 1024**3:.2f} GB "
      f"(cap {cfg['cap_gb']} GB)")

print("\nAFTER training (QLoRA-adapted, same prompts):")
model.eval()
model.config.use_cache = True
for r in rows:
    text = generate(model, r["prompt"])
    print(f'\n=== {r["id"]}  {r["type"]}  -> {verdict(r, text)}')
    print(text[:700])

In [ ]:
LEARN = dict(cap_gb=CAP_GB, model_id=MODEL_ID, data="train.jsonl", rank=16, lr=2e-4, epochs=8, max_new_tokens=350)
p = subprocess.run([sys.executable, "learn_check.py", json.dumps(LEARN)], capture_output=True, text=True,
                   env={**os.environ, "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})
print(p.stdout[-12000:])
if p.returncode:
    print("ERROR:\n", p.stderr[-3000:])

## 9. What to do with the results
- **Everything fits at "3 passages (real)" with `standard`:** the plan is feasible as written. Keep `liger` as a spare.
- **It fits only with `liger` or rank 16:** feasible, but write that setting into the training plan (Table 6) as a requirement.
- **"5 passages" fails but 3 passes:** that's evidence for the pending 3-vs-5 passages decision.
- **When the laptop is back:** run steps 4–7 on it with the same `CAP_GB` (or no cap) to confirm on the real card.

Before sharing this notebook, use `Edit → Clear all outputs`.